In [1]:
import os
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
base_dir = os.getcwd()

if os.path.basename(base_dir) == "notebooks":
    base_dir = os.path.dirname(base_dir)

data_dir = os.path.join(base_dir, "data")
image_dir = os.path.join(base_dir, "images")

os.makedirs(image_dir, exist_ok=True)

user_file = os.path.join(data_dir, "sample_user_behavior.csv")
item_file = os.path.join(data_dir, "sample_item_info.csv")

In [3]:
df = pd.read_csv(user_file)
user_stage = df.groupby("user_id").agg({
    "behavior_type": list
}).reset_index()

In [4]:
def load_data(user_file: str, item_file: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Load user behavior and item metadata."""
    user_df = pd.read_csv(user_file)
    item_df = pd.read_csv(item_file)

    required_user_cols = {
        "user_id", "item_id", "behavior_type", "user_geohash", "item_category", "time"
    }
    required_item_cols = {"item_id", "item_geohash", "item_category"}

    missing_user = required_user_cols - set(user_df.columns)
    missing_item = required_item_cols - set(item_df.columns)

    if missing_user:
        raise ValueError(f"Missing columns in user behavior data: {missing_user}")
    if missing_item:
        raise ValueError(f"Missing columns in item metadata: {missing_item}")

    return user_df, item_df

In [5]:
def has_action(actions, target):
    return int(target in actions)

user_stage["viewed"] = user_stage["behavior_type"].apply(lambda x: has_action(x, 1))
user_stage["cart"] = user_stage["behavior_type"].apply(lambda x: has_action(x, 3))
user_stage["purchase"] = user_stage["behavior_type"].apply(lambda x: has_action(x, 4))

view_users = user_stage["viewed"].sum()
cart_users = user_stage["cart"].sum()
purchase_users = user_stage["purchase"].sum()

print(view_users, cart_users, purchase_users)
BEHAVIOR_MAP = {
    1: "click",
    2: "collect",
    3: "add_to_cart",
    4: "purchase",
}

34 13 16


In [6]:
def preprocess_user_data(user_df: pd.DataFrame) -> pd.DataFrame:
    """Clean and preprocess user behavior data."""
    df = user_df.copy()

    df["time"] = pd.to_datetime(df["time"], errors="coerce")
    df = df.dropna(subset=["user_id", "item_id", "behavior_type", "item_category", "time"])

    df["behavior_type"] = df["behavior_type"].astype(int)
    df = df[df["behavior_type"].isin(BEHAVIOR_MAP.keys())]

    df["event_name"] = df["behavior_type"].map(BEHAVIOR_MAP)

    return df

In [7]:
def overall_funnel_by_user(user_df):
    stage_flags = user_df.pivot_table(
        index="user_id",
        columns="event_name",
        values="item_id",
        aggfunc="count",
        fill_value=0
    )

    viewed = stage_flags["click"] > 0
    added = stage_flags["add_to_cart"] > 0
    purchased = stage_flags["purchase"] > 0

    view_users = viewed.sum()
    cart_users = (viewed & added).sum()
    purchase_users = (viewed & added & purchased).sum()

    return pd.DataFrame({
        "stage": ["View (Click)", "Add to Cart", "Purchase"],
        "users": [view_users, cart_users, purchase_users]
    })

In [8]:
def compute_conversion_rates(funnel_df: pd.DataFrame) -> dict:
    """Compute stage conversion rates."""
    view_users = funnel_df.loc[funnel_df["stage"] == "View (Click)", "users"].iloc[0]
    cart_users = funnel_df.loc[funnel_df["stage"] == "Add to Cart", "users"].iloc[0]
    purchase_users = funnel_df.loc[funnel_df["stage"] == "Purchase", "users"].iloc[0]

    rates = {
        "view_to_cart_rate": round(cart_users / view_users * 100, 2) if view_users else 0,
        "cart_to_purchase_rate": round(purchase_users / cart_users * 100, 2) if cart_users else 0,
        "overall_conversion_rate": round(purchase_users / view_users * 100, 2) if view_users else 0,
    }
    return rates

In [9]:
def category_level_funnel(user_df: pd.DataFrame) -> pd.DataFrame:
    """Compute category-level funnel with sequential stage constraints."""
    grouped = (
        user_df.groupby(["user_id", "item_category", "event_name"])
        .size()
        .unstack(fill_value=0)
        .reset_index()
    )

    for col in ["click", "add_to_cart", "purchase"]:
        if col not in grouped.columns:
            grouped[col] = 0

    grouped["viewed"] = grouped["click"] > 0
    grouped["added_to_cart"] = grouped["add_to_cart"] > 0
    grouped["purchased"] = grouped["purchase"] > 0

    grouped["cart_stage"] = grouped["viewed"] & grouped["added_to_cart"]
    grouped["purchase_stage"] = grouped["viewed"] & grouped["added_to_cart"] & grouped["purchased"]

    category_df = (
        grouped.groupby("item_category")
        .agg(
            view_users=("viewed", "sum"),
            cart_users=("cart_stage", "sum"),
            purchase_users=("purchase_stage", "sum"),
        )
        .reset_index()
    )

    category_df["view_to_cart_rate"] = (
        category_df["cart_users"] / category_df["view_users"].replace(0, pd.NA) * 100
    ).round(2)

    category_df["cart_to_purchase_rate"] = (
        category_df["purchase_users"] / category_df["cart_users"].replace(0, pd.NA) * 100
    ).round(2)

    category_df["overall_conversion_rate"] = (
        category_df["purchase_users"] / category_df["view_users"].replace(0, pd.NA) * 100
    ).round(2)

    category_df = category_df.fillna(0)

    category_df = category_df.sort_values(
        by="overall_conversion_rate", ascending=False
    ).reset_index(drop=True)

    return category_df

In [10]:
def high_intent_non_converters(user_df: pd.DataFrame) -> pd.DataFrame:
    """Identify users with repeated clicks but no purchase."""
    user_event_counts = (
        user_df.groupby(["user_id", "event_name"])
        .size()
        .unstack(fill_value=0)
        .reset_index()
    )

    for col in ["click", "purchase"]:
        if col not in user_event_counts.columns:
            user_event_counts[col] = 0

    result = user_event_counts[
        (user_event_counts["click"] >= 5) & (user_event_counts["purchase"] == 0) &(user_event_counts["add_to_cart"] > 0)
    ].copy()

    result = result.sort_values(by="click", ascending=False)
    return result

In [11]:
def plot_funnel(funnel_df: pd.DataFrame, output_path: str) -> None:
    """Plot funnel chart."""
    plt.figure(figsize=(8, 5))
    plt.bar(funnel_df["stage"], funnel_df["users"])
    plt.title("Customer Conversion Funnel")
    plt.xlabel("Stage")
    plt.ylabel("Number of Users")
    plt.tight_layout()
    plt.savefig(output_path)
    plt.close()

In [12]:
def plot_category_conversion(category_df: pd.DataFrame, output_path: str) -> None:
    """Plot category-level overall conversion rates."""
    top_df = category_df.head(10)

    plt.figure(figsize=(10, 6))
    plt.bar(top_df["item_category"].astype(str), top_df["overall_conversion_rate"])
    plt.title("Top Categories by Overall Conversion Rate")
    plt.xlabel("Item Category")
    plt.ylabel("Overall Conversion Rate (%)")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig(output_path)
    plt.close()

In [13]:
user_df, item_df = load_data(user_file, item_file)
user_df = preprocess_user_data(user_df)
funnel_df = overall_funnel_by_user(user_df)
rates = compute_conversion_rates(funnel_df)
high_intent_df = high_intent_non_converters(user_df)

category_df = category_level_funnel(user_df)
category_df = category_df[category_df["view_users"] >= 5]
category_df = category_df.sort_values(
    by=["purchase_users", "overall_conversion_rate"],
    ascending=[False, False]
)

/var/folders/jg/4rd0rx294jj7ghlnsx6pp54h0000gn/T/ipykernel_79865/1549649859.py:43: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  category_df = category_df.fillna(0)


In [14]:
from IPython.display import display

print("=== Overall Funnel ===")
display(funnel_df)

print("\n=== Conversion Rates ===")
for k, v in rates.items():
    print(f"{k}: {v}%")

print("\n=== Category Funnel ===")
display(category_df.head(10))

print("\n=== High-intent Non-converters ===")
display(high_intent_df.head(10))

# plot
plot_funnel(funnel_df, os.path.join(image_dir, "funnel_chart.png"))
plot_category_conversion(category_df, os.path.join(image_dir, "category_chart.png"))

# save
funnel_df.to_csv(os.path.join(data_dir, "funnel_output.csv"), index=False)
category_df.to_csv(os.path.join(data_dir, "category_funnel_output.csv"), index=False)
high_intent_df.to_csv(os.path.join(data_dir, "high_intent_non_converters.csv"), index=False)

print(f"\nCharts saved to: {image_dir}")
print(f"Outputs saved to: {data_dir}")

=== Overall Funnel ===


,stage,users
0,View (Click),34
1,Add to Cart,13
2,Purchase,9



=== Conversion Rates ===
view_to_cart_rate: 38.24%
cart_to_purchase_rate: 69.23%
overall_conversion_rate: 26.47%

=== Category Funnel ===


,item_category,view_users,cart_users,purchase_users,view_to_cart_rate,cart_to_purchase_rate,overall_conversion_rate
5,6513,6,2,1,33.333333,50.0,16.666667
40,10392,5,1,0,20.000000,0.0,0.000000
149,13230,5,1,0,20.000000,0.0,0.000000
205,11552,5,0,0,0.000000,0.0,0.000000
216,12090,5,1,0,20.000000,0.0,0.000000
230,11623,5,1,0,20.000000,0.0,0.000000
267,3472,6,1,0,16.666667,0.0,0.000000
271,3381,6,1,0,16.666667,0.0,0.000000
290,1863,6,1,0,16.666667,0.0,0.000000
305,552,6,0,0,0.000000,0.0,0.000000



=== High-intent Non-converters ===


event_name,user_id,add_to_cart,click,collect,purchase
14,103582986,10,168,0,0
26,112296733,4,67,0,0
23,110253038,1,45,0,0
13,101781721,3,38,0,0



Charts saved to: /Users/user/Documents/github/Customer Conversion Funnel Analysis/images
Outputs saved to: /Users/user/Documents/github/Customer Conversion Funnel Analysis/data
